# README

## Перед прогоном ноутбука
В папке должны лежать датасеты:
1) train.parquet
2) benchmark_items.parquet
3) benchmark_queries.parquet

В этом ноутбуке EDA, гео-фильтр, эмбеддинги e5-base, валидация и бейзлайн (гео + косинус).
Финальное решение с бустингом — в `Avito_boosting_binary.ipynb`.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from google.colab import drive

# Посмотрим на данные

Хочется для начала проверить инфу
1) по категории
2) по флагу поиска с доставкой
3) по локации
4) по фильтрам поиска

Как-будто это должно сильно урезать выборку перед использованием семантики

In [4]:
train = pd.read_parquet('train.parquet')
bq    = pd.read_parquet('benchmark_queries.parquet')
items = pd.read_parquet('benchmark_items.parquet')

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

# Общее описание витрин

In [6]:
train.describe()

NameError: name 'train' is not defined

In [ ]:
bq.describe()

In [ ]:
items.describe()

# Теперь поработаем с данными

## 1) Проверим инфу по категориям


In [ ]:
train.groupby("search_category").count()

In [ ]:
bq.groupby("search_category").count()

In [ ]:
items.groupby("item_category_id").count()

Посмотрим глазами на эти категории

In [ ]:
bq[bq["search_category"] == 0].head(60)

In [ ]:
bq[bq["search_category"] == 114].head(50)

In [ ]:
items[items["item_category_id"] == 36].head(50)

Посмотрим теперь на микрокатегории items

In [ ]:
items.groupby("item_microcat_id").count().head(60)

## Инсайты по категориям:
1) В тренировочной выборке 114 категория доминирует над остальными, а в выборке для бенчмарков 0 категория начинает иметь больший процент, чем раньше (и в целом там только 2 категории - 0, 114)
2) По самому 0 номеру категории есть ощущение, что её просто не удалось правильно классифицировать (или это поиск без категории)
3) По названиям запросов в 114 категории видно, что это скорее всего "Услуги" + по тем запросам из 0 категории видно, что это тоже скорее всего "Услуги". Фактически датасет как раз из таких и состоит, а значит это поле не даст нам никакой информации
4) В items 1.18% предложений не из 114 категории, хотя по текстам там не редко проскакивают именно услуги -> оставим их

## 2) Проверим инфу по доставке

In [ ]:
train.groupby("search_is_delivery_search").count()

In [ ]:
bq.groupby("search_is_delivery_search").count()

## Инсайты по доставке:
1) Ну тут всё ещё проще, доставки практически нет, что логично, т.к. мы имеем дело с услугами

## 3) Проверим инфу по локации

In [ ]:
train['same_loc'] = train.search_location_id == train.item_location_id
train['same_loc'].mean()

np.float64(0.8310195650557696)

Заметим, что в достаточно приличном кол-ве случаев (83%) локация одна и та же, проверим по ширине и долготе насколько это рабочий метод (т.е. через расстояния)

In [ ]:
EARTH_RADIUS_KM = 6371


# расстояние по широте/долготе
def haversine_km(lat1, lon1, lat2, lon2):
    # в float64, иначе np.radians падает на object-колонках
    lat1, lon1, lat2, lon2 = (
        np.radians(np.asarray(x, dtype='float64')) for x in (lat1, lon1, lat2, lon2)
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


for col in ['item_latitude', 'item_longitude']:
    train[col] = pd.to_numeric(train[col], errors='coerce')

# центр локации поиска = медиана координат выбранных там объявлений
loc_centers = (
    train
    .groupby('search_location_id')
    .agg(lat=('item_latitude', 'median'),
         lon=('item_longitude', 'median'))
)
loc_centers.head(10)

train_geo = train.merge(loc_centers, left_on='search_location_id', right_index=True, how='left')

train_geo['dist_km'] = haversine_km(
    train_geo['lat'], train_geo['lon'],
    train_geo['item_latitude'], train_geo['item_longitude'],
)

is_same_loc = train_geo['search_location_id'] == train_geo['item_location_id']

print('Расстояние, когда объявление из той же локации:')
print(train_geo.loc[is_same_loc, 'dist_km'].describe(percentiles=[.5, .9, .99]))

print('\nРасстояние, когда объявление из другой локации:')
print(train_geo.loc[~is_same_loc, 'dist_km'].describe(percentiles=[.5, .8, .9, .95, .99]))

Расстояние, когда объявление из той же локации:
count    413573.000000
mean          5.144497
std          91.139828
min           0.000000
50%           2.585133
90%           9.829167
99%          20.427137
max       10972.693664
Name: dist_km, dtype: float64

Расстояние, когда объявление из другой локации:
count    84096.000000
mean       164.583345
std        572.197962
min          0.000000
50%         16.295821
80%         76.733148
90%        326.497769
95%        957.126118
99%       2911.549443
max      14773.725318
Name: dist_km, dtype: float64


Как-будто имеет смысл сделать гео-фильтр, через который будем отбирать пары (+ 5-10 мест можно оставить чисто для семантики, чтобы не пропустить что-то ну прям оч хорошее, но удаленное)

Ниже проверю насколько для одного location_id могут различаться координаты (в км)

In [ ]:
train_geo['is_same_loc'] = train_geo['search_location_id'] == train_geo['item_location_id']

# разброс выборов по каждой локации поиска
loc_spread = (
    train_geo
    .groupby('search_location_id')
    .agg(
        n_picks=('dist_km', 'size'),
        dist_p50=('dist_km', 'median'),
        dist_p90=('dist_km', lambda d: d.quantile(0.9)),
        share_far=('dist_km', lambda d: (d > 100).mean()),      # доля выборов дальше 100 км
        share_same_loc=('is_same_loc', 'mean'),
        n_item_locs=('item_location_id', 'nunique'),            # из скольких локаций выбирают
    )
)

loc_spread['n_bq'] = bq['search_location_id'].value_counts().reindex(loc_spread.index).fillna(0).astype(int)
loc_spread['has_own_items'] = loc_spread.index.isin(items['item_location_id'])

# где мало выборов - статистика шумная, пока не смотрим
MIN_PICKS = 30
reliable = loc_spread[loc_spread['n_picks'] >= MIN_PICKS]
print(f'Локаций всего: {len(loc_spread)}, с числом выборов >= {MIN_PICKS}: {len(reliable)}')

print('\nЛокации с наибольшей медианой расстояния:')
print(reliable.sort_values('dist_p50', ascending=False).head(20).round(2))

print('\nЛокации с наибольшей долей выборов дальше 100 км:')
print(reliable.sort_values('share_far', ascending=False).head(20).round(2))

Локаций всего: 2782, с числом выборов >= 30: 783

Локации с наибольшей медианой расстояния:
                    n_picks  dist_p50  dist_p90  share_far  share_same_loc  \
search_location_id                                                           
621540                 6839    639.20   2160.86       0.65            0.00   
660300                   55    173.61    233.71       0.53            0.00   
655190                  475     40.09    137.41       0.41            0.00   
637680                 1528     33.00     88.15       0.05            0.00   
636370                  434     32.69    108.61       0.11            0.00   
621550                  942     30.06     69.63       0.06            0.00   
629790                   33     15.69     23.12       0.00            0.48   
107620                12277     15.46     41.74       0.02            0.00   
645530                  103     14.16    101.66       0.22            0.00   
631080                  181     13.09    185.19   

Заметим, что есть локации с сильным разбросом:
621540 - как-будто вся Россия (для неё гео-фильтр оставим все локации)
другие - мб крупные регионы

Почекаем как разный радиус меняет покрытие

In [ ]:
# в items координаты приходят Decimal'ами, из-за этого падает median
coord_cols = ['item_latitude', 'item_longitude']
for df in (items, train):
    df[coord_cols] = df[coord_cols].astype('float64')

# центр локации объявлений - медиана координат её объявлений (корпус + train)
geo = pd.concat([items, train])[['item_location_id', 'item_latitude', 'item_longitude']]
item_loc_centers = geo.groupby('item_location_id')[['item_latitude', 'item_longitude']].median()
item_loc_centers.columns = ['lat', 'lon']

# центр локации поиска: если есть такая локация объявлений - её центр, иначе (регионы) - по выборам из train
search_loc_centers = pd.concat([item_loc_centers, loc_centers.astype('float64')])
search_loc_centers = search_loc_centers[~search_loc_centers.index.duplicated(keep='first')]

# расстояние между центрами локации поиска и локации объявления для каждой пары train
pairs = train[['search_location_id', 'item_location_id']].copy()
pairs = pairs.join(search_loc_centers, on='search_location_id')
pairs = pairs.join(item_loc_centers, on='item_location_id', rsuffix='_item')
pairs['loc_dist_km'] = haversine_km(pairs['lat'], pairs['lon'], pairs['lat_item'], pairs['lon_item'])

# сколько объявлений корпуса попадает в радиус для запросов бенча
corpus_locs = items['item_location_id'].value_counts().rename('n_items').to_frame().join(item_loc_centers)
bq_locs = search_loc_centers.reindex(bq['search_location_id'].unique())

# матрица расстояний: локации поиска бенча x локации корпуса
dist_matrix = haversine_km(
    bq_locs['lat'].values[:, None], bq_locs['lon'].values[:, None],
    corpus_locs['lat'].values[None, :], corpus_locs['lon'].values[None, :],
)

rows = []
for R in [1, 5, 10, 20, 30, 40, 50, 80, 120, 200]:
    coverage = (pairs['loc_dist_km'] <= R).mean()

    pool_by_loc = pd.Series((dist_matrix <= R) @ corpus_locs['n_items'].values, index=bq_locs.index)
    bq_pool = bq['search_location_id'].map(pool_by_loc)

    rows.append({'R_km': R, 'покрытие_train': round(coverage, 3),
                 'пул_медиана': int(bq_pool.median()), 'пул_p90': int(bq_pool.quantile(0.9))})

pd.DataFrame(rows)

,R_km,покрытие_train,пул_медиана,пул_p90
0,1,0.885,2326,19543
1,5,0.895,2326,19543
2,10,0.901,2369,19543
3,20,0.924,2890,22855
4,30,0.940,3274,27537
5,40,0.951,3450,31651
6,50,0.956,3533,32754
7,80,0.966,4020,36589
8,120,0.973,4507,40904
9,200,0.980,7716,47558


R = 40 даёт хорошее покрытие, возьмем такой для гео-фильтра

In [ ]:
R_KM = 40
COUNTRY_P50_KM = 300   # медиана расстояния больше - считаем локацию "всей страной"
MIN_PICKS = 30

# не хардкожу 621540, а ловлю правилом
country_locs = set(
    loc_spread.index[(loc_spread['n_picks'] >= MIN_PICKS) & (loc_spread['dist_p50'] > COUNTRY_P50_KM)]
)
print('Локации без гео-фильтра:', country_locs)


def build_geo_filter(radius_km, search_centers, item_centers, no_filter_locs):
    # {search_location_id: set(item_location_id в радиусе)}, None = фильтр не применяем
    search_centers = search_centers.dropna()

    dist = haversine_km(
        search_centers['lat'].values[:, None], search_centers['lon'].values[:, None],
        item_centers['lat'].values[None, :], item_centers['lon'].values[None, :],
    )
    item_loc_ids = item_centers.index.to_numpy()

    geo_filter = {}
    for row, loc_id in enumerate(search_centers.index):
        if loc_id in no_filter_locs:
            geo_filter[loc_id] = None
            continue

        in_radius = item_loc_ids[dist[row] <= radius_km]
        # никого в радиусе - берём ближайшую, чтобы пул не был пустым
        if len(in_radius) == 0:
            in_radius = item_loc_ids[[dist[row].argmin()]]

        geo_filter[loc_id] = set(in_radius)
    return geo_filter


search_locs = pd.Index(pd.concat([train['search_location_id'], bq['search_location_id']]).unique())

geo_filter = build_geo_filter(
    radius_km=R_KM,
    search_centers=search_loc_centers.reindex(search_locs),
    item_centers=item_loc_centers,
    no_filter_locs=country_locs,
)
print('Локаций в гео-фильтре:', len(geo_filter))


def get_geo_pool(search_location_id):
    return geo_filter.get(search_location_id)

Локации без гео-фильтра: {621540}
Локаций в гео-фильтре: 2782


In [ ]:
def is_in_geo_pool(search_location_id, item_location_id):
    pool = get_geo_pool(search_location_id)
    return pool is None or item_location_id in pool

# покрытие на train - доля выборов, которые проходят фильтр
covered = [is_in_geo_pool(s, i) for s, i in zip(train['search_location_id'], train['item_location_id'])]
print(f'Покрытие train при R={R_KM} км: {np.mean(covered):.3f}')

corpus_loc_counts = items['item_location_id'].value_counts()

def geo_pool_size(search_location_id):
    pool = get_geo_pool(search_location_id)
    if pool is None:
        return len(items)
    return int(corpus_loc_counts.reindex(list(pool)).sum())   # локации без объявлений в корпусе -> NaN, sum их скипает

bq_pool_size = bq['search_location_id'].map(geo_pool_size)
print('Размер пула в бенчмарке:')
print(bq_pool_size.describe(percentiles=[.25, .5, .75, .9]).astype(int))
print('Запросов без гео-фильтра:', bq['search_location_id'].map(get_geo_pool).isna().sum())

Покрытие train при R=40 км: 0.960
Размер пула в бенчмарке:
count      2452
mean      16657
std       36406
min          39
25%        1212
50%        3450
75%       14600
90%       31651
max      189212
Name: search_location_id, dtype: int64
Запросов без гео-фильтра: 94


Отлично, теперь есть гео-фильтр с хорошим покрытием, который сильно урезает кол-во объявлений для вывода (3450 по медиане, 16.6к по среднему)


geo_filter - сам гео-фильтр (словарь кароч)

get_geo_pool(search_location_id) - функция для доступа к нему

# Само решение
Теперь попробуем собрать выборку для отправки (answer.csv)

Будем смотреть по семантическому сходству через косинусное сходство эмбеддингов
* 40-45 с гео-фильтром
* 5-10 без гео-фильтра, чтобы не пропустить редкие далекие, но полезные




## Эмбеддинги

Модель — `intfloat/multilingual-e5-base` (open-source, с HF), префиксы `query:` / `passage:` как в оригинале.

- объявления кодирую один раз: корпус + объявления из train без дублей по item_id, одна матрица и для бенча, и для валидации
- запросы — по уникальным (текст, фильтры), сразу в двух вариантах: с фильтрами и без, потом сравню
- текст объявления = заголовок + параметры + первые 500 символов описания (дальше обычно реклама)

In [ ]:
drive.mount('/content/drive')

# храню на драйве, чтобы не пересчитывать после рестарта
EMB_DIR = '/content/drive/MyDrive/avito_bootcamp/emb_e5_base'
os.makedirs(EMB_DIR, exist_ok=True)

MODEL_NAME = 'intfloat/multilingual-e5-base'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device)

model = SentenceTransformer(MODEL_NAME, device=device)
model = model.half()   # fp16, на T4 в разы быстрее

Mounted at /content/drive
Устройство: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (5 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
MAX_DESC_CHARS = 500


def clean_text(text):
    # чистим эмодзи/спецсимволы и лишние пробелы
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ''
    text = str(text)
    text = re.sub(r'[^\w\s.,:;!?()«»"\'/+%-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def make_query_text(search_query, params_text, with_params):
    text = clean_text(search_query)
    params = clean_text(params_text)
    if with_params and params:
        text = f'{text}. {params}'
    return f'query: {text}'


def make_item_text(title, params_text, description):
    # порядок такой, чтобы при обрезке по токенам терялся конец описания, а не заголовок
    parts = [
        clean_text(title),
        clean_text(params_text),
        clean_text(description)[:MAX_DESC_CHARS],
    ]
    return 'passage: ' + '. '.join(p for p in parts if p)


row = train.iloc[0]
print(make_query_text(row['search_query'], row['search_infm_params_text'], with_params=True))
print(make_item_text(row['item_title_raw'], row['item_infm_params_text'], row['item_description_raw']))

query: скупка телевизоров
passage: Скупка б/у техники. Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за услугу Работаете с юрлицами и ИП Опыт работы 10 лет и больше Гарантия на выполнение работ Как вы работаете У себя Предоплата Рабочие дни Понедельник Рабочие дни Вторник Рабочие дни Среда Рабочие дни Четверг Рабочие дни Пятница Рабочие дни Суббота Рабочие дни Воскресенье Выезд к клиенту. Скупаю практически любую современную новую и б/у технику(обязательно рабочую), до 80% от рыночной стоимости. Все предложения присылайте в личные сообщения Авито, с указанием точной модели и фото. Строго от 18 лет, через документы удостоверяющие личность. График работы 24/7.


In [ ]:
ITEM_COLS = ['item_id', 'item_title_raw', 'item_infm_params_text', 'item_description_raw']
QUERY_COLS = ['search_query', 'search_infm_params_text']

all_items = (
    pd.concat([items[ITEM_COLS], train[ITEM_COLS]])
    .drop_duplicates('item_id')
    .reset_index(drop=True)
)
all_items['in_corpus'] = all_items['item_id'].isin(items['item_id'])
all_items['text'] = [
    make_item_text(t, p, d)
    for t, p, d in zip(all_items['item_title_raw'],
                       all_items['item_infm_params_text'],
                       all_items['item_description_raw'])
]

all_queries = (
    pd.concat([train[QUERY_COLS], bq[QUERY_COLS]])
    .drop_duplicates()
    .reset_index(drop=True)
)
all_queries['text_no_params'] = [
    make_query_text(q, p, with_params=False)
    for q, p in zip(all_queries['search_query'], all_queries['search_infm_params_text'])
]
all_queries['text_with_params'] = [
    make_query_text(q, p, with_params=True)
    for q, p in zip(all_queries['search_query'], all_queries['search_infm_params_text'])
]

print(f'Уникальных объявлений: {len(all_items)} (из них в корпусе: {all_items["in_corpus"].sum()})')
print(f'Уникальных запросов: {len(all_queries)}')

Уникальных объявлений: 515895 (из них в корпусе: 189212)
Уникальных запросов: 109736


In [ ]:
def encode_and_save(texts, name, max_seq_length, batch_size=128):
    # если уже посчитано - просто грузим с диска
    path = os.path.join(EMB_DIR, f'{name}.npy')
    if os.path.exists(path):
        print(f'{name}: загружаю готовые эмбеддинги')
        return np.load(path)

    model.max_seq_length = max_seq_length
    emb = model.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,   # нормируем -> косинус = dot product
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    emb = emb.astype(np.float16)   # в 2 раза меньше места, для поиска точности хватает
    np.save(path, emb)
    print(f'{name}: сохранено, форма {emb.shape}')
    return emb



# индексы рядом: номер строки в parquet = номер строки в npy
all_items[['item_id', 'in_corpus']].to_parquet(os.path.join(EMB_DIR, 'items_index.parquet'))
all_queries[QUERY_COLS].to_parquet(os.path.join(EMB_DIR, 'queries_index.parquet'))

# запросам хватает 64 токенов, объявлениям 128 (на 256 было сильно дольше)
print('\nЗапросы (без фильтров)')
query_emb_no_params = encode_and_save(all_queries['text_no_params'], 'queries_no_params', max_seq_length=64)

print('\nЗапросы (с фильтрами)')
query_emb_with_params = encode_and_save(all_queries['text_with_params'], 'queries_with_params', max_seq_length=64)

print('\nВсе объявления (корпус + train)')
item_emb = encode_and_save(all_items['text'], 'items', max_seq_length=128)


Запросы (без фильтров)
queries_no_params: загружаю готовые эмбеддинги

Запросы (с фильтрами)
queries_with_params: загружаю готовые эмбеддинги

Все объявления (корпус + train)
items: загружаю готовые эмбеддинги


In [ ]:
# только корпус - для поиска по бенчу
corpus_mask = all_items['in_corpus'].to_numpy()
corpus_item_emb = item_emb[corpus_mask]
corpus_item_ids = all_items.loc[corpus_mask, 'item_id'].to_numpy()

## Валидация

Для бенча разметки нет, а попыток всего 7 — нужна своя валидация на train.

In [ ]:
# в train нет query_id, поэтому запрос = группа строк с одинаковыми search_*
SEARCH_COLS = ['search_query', 'search_location_id', 'search_is_delivery_search',
               'search_infm_params_text', 'search_category']

train['query_group'] = train.groupby(SEARCH_COLS, dropna=False).ngroup()

groups = train.groupby('query_group').agg(
    n_rows=('item_id', 'size'),
    n_items=('item_id', 'nunique'),
)

print(f'Строк в train: {len(train)}')
print(f'Групп (запросов): {len(groups)}')
print(f'Групп, где одно объявление встречается несколько раз: {(groups["n_rows"] > groups["n_items"]).mean():.3f}')

print('\nСколько объявлений в группе (доля групп):')
size_dist = groups['n_items'].value_counts(normalize=True).sort_index()
print(size_dist.head(10).round(4))
print(f'Больше 10 объявлений: {(groups["n_items"] > 10).mean():.4f}')

# самые жирные группы - глазами
biggest = groups.sort_values('n_items', ascending=False).head(10)
print('\nСамые крупные группы:')
print(
    train.drop_duplicates('query_group')
    .set_index('query_group')
    .loc[biggest.index, ['search_query', 'search_location_id', 'search_infm_params_text']]
    .join(biggest)
)

Строк в train: 497673
Групп (запросов): 354463
Групп, где одно объявление встречается несколько раз: 0.039

Сколько объявлений в группе (доля групп):
n_items
1     0.8500
2     0.0949
3     0.0264
4     0.0111
5     0.0057
6     0.0034
7     0.0021
8     0.0015
9     0.0011
10    0.0007
Name: proportion, dtype: float64
Больше 10 объявлений: 0.0032

Самые крупные группы:
                        search_query  search_location_id  \
query_group                                                
127140                       маникюр              637640   
127726                       маникюр              653240   
155854            наращивание ресниц              637640   
156446            наращивание ресниц              653240   
324253       установка кондиционеров              637640   
127716                       маникюр              653240   
127132                       маникюр              637640   
178206                       педикюр              637640   
131725                     

In [ ]:
def norm(s):
    return s.fillna('').str.lower().str.replace('ё', 'е').str.strip()


N_VAL = 2500           # примерно как бенч
KNOWN_SHARE = 0.37     # в бенче 37% текстов запросов встречаются в train
RANDOM_STATE = 42

group_info = (
    train.drop_duplicates('query_group')
    .set_index('query_group')[['search_query']]
)
group_info['text_norm'] = norm(group_info['search_query'])

# "знакомый" = текст есть ещё в других группах, т.е. останется в train после отбора
text_n_groups = group_info['text_norm'].map(group_info['text_norm'].value_counts())
group_info['is_known'] = text_n_groups > 1

print(f'Групп со «знакомым» текстом: {group_info["is_known"].mean():.3f}')

# если брать рандомно, будет 87% знакомых и валидация выйдет слишком лёгкой -> стратифицирую под бенч
n_known = int(N_VAL * KNOWN_SHARE)
val_known = group_info[group_info['is_known']].sample(n_known, random_state=RANDOM_STATE)
val_new = group_info[~group_info['is_known']].sample(N_VAL - n_known, random_state=RANDOM_STATE)
val_groups = set(val_known.index) | set(val_new.index)

is_val = train['query_group'].isin(val_groups)
val = train[is_val]
train_rest = train[~is_val]

print(f'Валидация: {len(val_groups)} запросов, {len(val)} строк')
print(f'Остаток train: {train_rest["query_group"].nunique()} запросов, {len(train_rest)} строк')

known_texts = set(norm(train_rest['search_query']))
val_texts = norm(val.drop_duplicates('query_group')['search_query'])
print(f'Доля «знакомых» текстов в валидации: {val_texts.isin(known_texts).mean():.3f} (в бенчмарке 0.370)')

Групп со «знакомым» текстом: 0.868
Валидация: 2500 запросов, 2956 строк
Остаток train: 351963 запросов, 494717 строк
Доля «знакомых» текстов в валидации: 0.370 (в бенчмарке 0.370)


In [ ]:
val_answers = val.groupby('query_group')['item_id'].apply(set)
print('Правильных объявлений на запрос:')
print(val_answers.map(len).value_counts().sort_index().head(10))

# сохраняю сплит, чтобы в ноутбуке с бустингом была та же валидация
pd.Series(sorted(val_groups), name='query_group').to_frame().to_parquet(
    os.path.join(EMB_DIR, 'val_groups.parquet')
)

Правильных объявлений на запрос:
item_id
1     2304
2      133
3       37
4       12
5        8
6        2
8        1
13       1
20       1
30       1
Name: count, dtype: int64


In [ ]:
val_answer_items = set().union(*val_answers)

# корпус для валидации = корпус бенча + правильные ответы валидации (иначе их просто нет в корпусе)
search_mask = (all_items['in_corpus'] | all_items['item_id'].isin(val_answer_items)).to_numpy()
search_item_ids = all_items.loc[search_mask, 'item_id'].to_numpy()
search_emb = item_emb[search_mask].astype(np.float32)

item_location = (
    pd.concat([items[['item_id', 'item_location_id']], train[['item_id', 'item_location_id']]])
    .drop_duplicates('item_id')
    .set_index('item_id')['item_location_id']
)
search_item_locs = item_location.loc[search_item_ids].to_numpy()

print(f'Объявлений в корпусе для валидации: {len(search_item_ids)} '
      f'(добавлено из train: {len(search_item_ids) - all_items["in_corpus"].sum()})')

Объявлений в корпусе для валидации: 191887 (добавлено из train: 2675)


In [ ]:
val_queries = (
    val.drop_duplicates('query_group')
    .set_index('query_group')
    .loc[val_answers.index, SEARCH_COLS]
)

query_row = pd.Series(all_queries.index, index=pd.MultiIndex.from_frame(all_queries[QUERY_COLS]))
val_query_rows = query_row.loc[pd.MultiIndex.from_frame(val_queries[QUERY_COLS])].to_numpy()

val_q_emb = {
    'без фильтров': query_emb_no_params[val_query_rows].astype(np.float32),
    'с фильтрами': query_emb_with_params[val_query_rows].astype(np.float32),
}
val_q_locs = val_queries['search_location_id'].to_numpy()

### Бейзлайн: гео-фильтр + косинус

In [ ]:
_geo_mask_cache = {}

def get_geo_mask(search_location_id):
    # маска по корпусу, кэшим т.к. локации повторяются
    if search_location_id not in _geo_mask_cache:
        pool = get_geo_pool(search_location_id)
        if pool is None:
            mask = np.ones(len(search_item_locs), dtype=bool)
        else:
            mask = np.isin(search_item_locs, list(pool))
        _geo_mask_cache[search_location_id] = mask
    return _geo_mask_cache[search_location_id]


def search_top_k(query_emb, query_locs, use_geo, k=500, batch_size=256):
    # топ-k по косинусу, батчами чтобы не упасть по памяти
    top = np.empty((len(query_emb), k), dtype=np.int64)
    for start in range(0, len(query_emb), batch_size):
        end = start + batch_size
        scores = query_emb[start:end] @ search_emb.T

        if use_geo:
            for row, loc_id in enumerate(query_locs[start:end]):
                scores[row, ~get_geo_mask(loc_id)] = -np.inf

        # argpartition + сортировка только топа - быстрее полной сортировки
        part = np.argpartition(-scores, k, axis=1)[:, :k]
        part_scores = np.take_along_axis(scores, part, axis=1)
        order = np.argsort(-part_scores, axis=1)
        top[start:end] = np.take_along_axis(part, order, axis=1)
    return top


def recall_at_k(top_idx, answers, k):
    recalls = []
    for row, true_items in enumerate(answers):
        found = set(search_item_ids[top_idx[row, :k]])
        recalls.append(len(found & true_items) / len(true_items))
    return np.mean(recalls)

In [ ]:
results = []
for query_variant, q_emb in val_q_emb.items():
    for use_geo in [False, True]:
        top = search_top_k(q_emb, val_q_locs, use_geo=use_geo)
        row = {'запрос': query_variant, 'гео-фильтр': use_geo}
        for k in [50, 100, 200, 500]:
            row[f'recall@{k}'] = round(recall_at_k(top, val_answers.to_numpy(), k), 4)
        results.append(row)

pd.DataFrame(results)

In [ ]:
# переписал поиск: корпус теперь передаётся явно (маски привязаны к порядку объявлений в корпусе,
# а для бенча корпус другой) + отдельно глобальный топ без гео
def build_geo_masks(corpus_locs, query_locs):
    masks = {}
    for loc_id in np.unique(query_locs):
        pool = get_geo_pool(loc_id)
        if pool is None:
            masks[loc_id] = np.ones(len(corpus_locs), dtype=bool)
        else:
            masks[loc_id] = np.isin(corpus_locs, list(pool))
    return masks


def search_top_k(query_emb, corpus_emb, k, geo_masks=None, query_locs=None, batch_size=256):
    top_idx = np.empty((len(query_emb), k), dtype=np.int64)
    top_scores = np.empty((len(query_emb), k), dtype=np.float32)

    for start in range(0, len(query_emb), batch_size):
        end = start + batch_size
        scores = query_emb[start:end] @ corpus_emb.T

        if geo_masks is not None:
            for row, loc_id in enumerate(query_locs[start:end]):
                scores[row, ~geo_masks[loc_id]] = -np.inf

        part = np.argpartition(-scores, k, axis=1)[:, :k]
        part_scores = np.take_along_axis(scores, part, axis=1)
        order = np.argsort(-part_scores, axis=1)
        top_idx[start:end] = np.take_along_axis(part, order, axis=1)
        top_scores[start:end] = np.take_along_axis(part_scores, order, axis=1)

    return top_idx, top_scores


def merge_candidates(geo_idx, geo_scores, global_idx, n_total=50, n_global=5):
    # сначала лучшие из гео-пула (n_total - n_global), потом добиваем глобальным топом
    result = []
    for idx, score in zip(geo_idx, geo_scores):
        if len(result) == n_total - n_global:
            break
        if np.isfinite(score):          # -inf = вне гео-пула
            result.append(idx)

    for idx in global_idx:
        if len(result) == n_total:
            break
        if idx not in result:
            result.append(idx)
    return result


def recall_at_50(pred_item_ids, answers):
    return np.mean([len(set(pred) & true) / len(true) for pred, true in zip(pred_item_ids, answers)])

In [ ]:
q_emb = val_q_emb['с фильтрами']
val_geo_masks = build_geo_masks(search_item_locs, val_q_locs)

geo_idx, geo_scores = search_top_k(q_emb, search_emb, k=50, geo_masks=val_geo_masks, query_locs=val_q_locs)
global_idx, _ = search_top_k(q_emb, search_emb, k=50)

# сколько мест отдавать объявлениям без гео
for n_global in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    preds = [
        search_item_ids[merge_candidates(g_idx, g_sc, gl_idx, n_global=n_global)]
        for g_idx, g_sc, gl_idx in zip(geo_idx, geo_scores, global_idx)
    ]
    print(f'Мест без гео-фильтра: {n_global:2d} | recall@50 = {recall_at_50(preds, val_answers):.4f}')

Мест без гео-фильтра:  0 | recall@50 = 0.8016
Мест без гео-фильтра:  1 | recall@50 = 0.8034
Мест без гео-фильтра:  2 | recall@50 = 0.8046
Мест без гео-фильтра:  3 | recall@50 = 0.8048
Мест без гео-фильтра:  4 | recall@50 = 0.8040
Мест без гео-фильтра:  5 | recall@50 = 0.8027
Мест без гео-фильтра:  6 | recall@50 = 0.7997
Мест без гео-фильтра:  7 | recall@50 = 0.8005
Мест без гео-фильтра:  8 | recall@50 = 0.7996
Мест без гео-фильтра:  9 | recall@50 = 0.7983
Мест без гео-фильтра: 10 | recall@50 = 0.7972


3 места без гео дают небольшой плюс (0.8016 → 0.8048), дальше уже хуже. Оставляю N_GLOBAL = 3.

In [ ]:
N_GLOBAL = 3
RADII = [10, 20, 25, 30, 35, 40, 60, 80, 120]

q_emb = val_q_emb['с фильтрами']

# глобальный топ от R не зависит
global_idx, _ = search_top_k(q_emb, search_emb, k=50)

radius_results = []
for radius_km in RADII:
    # get_geo_pool смотрит в глобальный geo_filter, поэтому просто перезаписываю его
    geo_filter = build_geo_filter(
        radius_km=radius_km,
        search_centers=search_loc_centers.reindex(search_locs),
        item_centers=item_loc_centers,
        no_filter_locs=country_locs,
    )
    val_geo_masks = build_geo_masks(search_item_locs, val_q_locs)

    geo_idx, geo_scores = search_top_k(q_emb, search_emb, k=50,
                                       geo_masks=val_geo_masks, query_locs=val_q_locs)
    preds = [
        search_item_ids[merge_candidates(g_idx, g_sc, gl_idx, n_global=N_GLOBAL)]
        for g_idx, g_sc, gl_idx in zip(geo_idx, geo_scores, global_idx)
    ]

    pool_sizes = [val_geo_masks[loc].sum() for loc in val_q_locs]

    radius_results.append({
        'R_km': radius_km,
        'recall@50': round(recall_at_50(preds, val_answers), 4),
        'пул_медиана': int(np.median(pool_sizes)),
    })
    print(radius_results[-1])

pd.DataFrame(radius_results)

{'R_km': 10, 'recall@50': np.float64(0.7965), 'пул_медиана': 1638}
{'R_km': 20, 'recall@50': np.float64(0.7991), 'пул_медиана': 1916}
{'R_km': 25, 'recall@50': np.float64(0.8042), 'пул_медиана': 2144}
{'R_km': 30, 'recall@50': np.float64(0.8072), 'пул_медиана': 2144}
{'R_km': 35, 'recall@50': np.float64(0.808), 'пул_медиана': 2248}
{'R_km': 40, 'recall@50': np.float64(0.8048), 'пул_медиана': 2252}
{'R_km': 60, 'recall@50': np.float64(0.8023), 'пул_медиана': 2355}
{'R_km': 80, 'recall@50': np.float64(0.793), 'пул_медиана': 2980}
{'R_km': 120, 'recall@50': np.float64(0.7787), 'пул_медиана': 4380}


,R_km,recall@50,пул_медиана
0,10,0.7965,1638
1,20,0.7991,1916
2,25,0.8042,2144
3,30,0.8072,2144
4,35,0.8080,2248
5,40,0.8048,2252
6,60,0.8023,2355
7,80,0.7930,2980
8,120,0.7787,4380


Между 25 и 60 км разница в пределах шума (0.804–0.808), 80+ уже проседает. Оставил 40, как было по покрытию.

# Генерация итогового ответа

Выше игрался с гиперпараметрами, чтобы понять наилучшие. Дальше уже с выбранными гиперпараметрами делаем сам итоговый ответ

In [ ]:
# проверка формата по условию, чтобы не слить попытку
def check_answer(answer_df):
    valid_ids = set(items['item_id'])
    assert list(answer_df.columns) == ['query_id', 'answer'], 'Колонки должны быть ровно query_id и answer'
    assert answer_df['query_id'].is_unique, 'Есть повторяющиеся query_id'
    assert set(answer_df['query_id']) == set(bq['query_id']), 'Набор query_id не совпадает с бенчмарком'
    for ans in answer_df['answer']:
        ids = ans.split(' ')
        assert len(ids) <= 50, 'Больше 50 объявлений в строке'
        assert len(ids) == len(set(ids)), 'Повторы объявлений внутри строки'
        assert all(i in valid_ids for i in ids), 'Есть item_id не из корпуса'
    print('Формат в порядке. Средняя длина ответа:', answer_df['answer'].str.split().str.len().mean())

## Первый сабмит: только семантика + гео-фильтр

R = 40, N_GLOBAL = 3. На лидерборде вышло 0.75 при 0.805 на валидации — валидация чуть оптимистичнее (видимо, в корпусе бенча плотнее похожие объявления вокруг правильных), но в целом ей можно верить для сравнения вариантов.

In [ ]:
N_GLOBAL = 3   # по валидации

# для бенча корпус - только benchmark_items
corpus_mask = all_items['in_corpus'].to_numpy()
corpus_ids = all_items.loc[corpus_mask, 'item_id'].to_numpy()
corpus_emb = item_emb[corpus_mask].astype(np.float32)
corpus_locs = item_location.loc[corpus_ids].to_numpy()

bq_query_rows = query_row.loc[pd.MultiIndex.from_frame(bq[QUERY_COLS])].to_numpy()
bq_q_emb = query_emb_with_params[bq_query_rows].astype(np.float32)
bq_q_locs = bq['search_location_id'].to_numpy()

bq_geo_masks = build_geo_masks(corpus_locs, bq_q_locs)
bq_geo_idx, bq_geo_scores = search_top_k(bq_q_emb, corpus_emb, k=50, geo_masks=bq_geo_masks, query_locs=bq_q_locs)
bq_global_idx, _ = search_top_k(bq_q_emb, corpus_emb, k=50)

predictions = [
    corpus_ids[merge_candidates(g_idx, g_sc, gl_idx, n_global=N_GLOBAL)]
    for g_idx, g_sc, gl_idx in zip(bq_geo_idx, bq_geo_scores, bq_global_idx)
]

answer = pd.DataFrame({
    'query_id': bq['query_id'].to_numpy(),
    'answer': [' '.join(pred) for pred in predictions],
})

In [ ]:
check_answer(answer)

answer_path = os.path.join(EMB_DIR, 'answer_baseline_geo_e5.csv')
answer.to_csv(answer_path, index=False)

from google.colab import files
files.download(answer_path)